In [0]:
import os
import sys

In [0]:
repo_root = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(repo_root, "src"))

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
 
from data_health_ftw.common import config
from data_health_ftw.ml.dataset import build_modeling_frame

In [0]:
RANDOM_STATE = 42
N_SPLITS = 5

In [0]:
raw = spark.read.table(config.GOLD_ML_FEATURES_TABLE).toPandas()
print(f"rows={len(raw)}, countries={raw['country_code'].nunique()}, years={sorted(int(y) for y in raw['year'].unique())}")
n_duplicates = raw.duplicated(config.GOLD_ML_UNIQUE_KEY).sum()
if n_duplicates > 0:
    raise ValueError(f"{n_duplicates} duplicated (country_code, year) keys")

In [0]:
null_by_year = (
    raw.groupby("year")[config.INDICATOR_NAMES]
    .apply(lambda g: g.isna().mean())
    .mul(100)
    .round(1)
)
null_by_year.T

In [0]:
features = list(config.ML_FEATURES)
targets = list(config.ML_TARGETS)
 
frame = build_modeling_frame(
    raw,
    features=features,
    targets=targets,
    min_year=config.ML_MIN_YEAR,
    fill_from_past_columns=config.ML_FILL_FROM_PAST,
)
print(f"modelling rows={len(frame)}, countries={frame['country_code'].nunique()}")
frame[features].isna().mean().mul(100).round(1).sort_values(ascending=False)

In [0]:
fig, axes = plt.subplots(2, len(targets), figsize=(5 * len(targets), 7))
for j, target in enumerate(targets):
    axes[0, j].hist(frame[target], bins=40)
    axes[0, j].set_title(f"{target} (raw)")
    axes[1, j].hist(np.log1p(frame[target]), bins=40)
    axes[1, j].set_title(f"log1p({target})")
plt.tight_layout()
plt.show()
 
frame[targets].describe().round(1)

In [0]:
for target in targets:
    y = np.log1p(frame[target])
    total = ((y - y.mean()) ** 2).sum()
    by_country = ((frame.assign(y=y).groupby("country_code")["y"].transform("mean") - y.mean()) ** 2).sum()
    by_year = ((frame.assign(y=y).groupby("year")["y"].transform("mean") - y.mean()) ** 2).sum()
    print(f"{target:20s} country={by_country / total:5.1%}  year={by_year / total:5.1%}")

In [0]:
corr = frame[features + targets].corr(method="spearman").loc[features, targets].round(2)
display(corr)
 
fig, ax = plt.subplots(figsize=(4 + len(targets), 5))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(targets)), targets, rotation=45, ha="right")
ax.set_yticks(range(len(features)), features)
fig.colorbar(im)
plt.tight_layout()
plt.show()

In [0]:
display(corr.reset_index(names="feature"))